# Sentiment Analysis & Classification

Exploration and prototyping for the sentiment analysis and classification components.

In [ ]:
!pip install -q transformers torch datasets scikit-learn pandas

In [ ]:
from datasets import load_dataset

ds = load_dataset('fancyzhx/amazon_polarity', split='train', streaming=True)
sample = []
for i, row in enumerate(ds):
    if i >= 200:
        break
    sample.append(row)

texts = [r['content'] for r in sample]
print(texts[:3])

README.md:   0%|          | 0.00/6.81k [00:00<?, ?B/s]

['This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I have ever played it has the best music! It backs away from crude keyboarding and takes a fresher step with grate guitars and soulful orchestras. It would impress anyone who cares to listen! ^_^', "I'm reading a lot of reviews saying that this is the best 'game soundtrack' and I figured that I'd write a review to disagree a bit. This in my opinino is Yasunori Mitsuda's ultimate masterpiece. The music is timeless and I'm been listening to it for years now and its beauty simply refuses to fade.The price tag on this is pretty staggering I must say, but if you are going to buy any cd for this much money, this is the only one that I feel would be worth every penny.", 'This soundtrack is my favorite music of all time, hands down. The intense sadness of "Prisoners of Fate" (which means all the 

In [ ]:
from transformers import pipeline

sentiment_pipe = pipeline('sentiment-analysis')
results = sentiment_pipe(texts[:20], truncation=True)
for t, r in zip(texts[:20], results):
    print(r['label'], round(r['score'], 2), '--', t[:80])

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

POSITIVE 1.0 -- This sound track was beautiful! It paints the senery in your mind so well I woul
POSITIVE 0.99 -- I'm reading a lot of reviews saying that this is the best 'game soundtrack' and 
POSITIVE 1.0 -- This soundtrack is my favorite music of all time, hands down. The intense sadnes
POSITIVE 1.0 -- I truly like this soundtrack and I enjoy video game music. I have played this ga
POSITIVE 1.0 -- If you've played the game, you know how divine the music is! Every single song t
POSITIVE 1.0 -- I am quite sure any of you actually taking the time to read this have played the
NEGATIVE 1.0 -- This is a self-published book, and if you want to know why--read a few paragraph
POSITIVE 1.0 -- I loved Whisper of the wicked saints. The story was amazing and I was pleasantly
POSITIVE 1.0 -- I just finished reading Whisper of the Wicked saints. I fell in love with the ca
POSITIVE 1.0 -- This was a easy to read book that made me want to keep reading on and on, not ea
NEGATIVE 1.0 -- A complete wa

## Classification prototyping

Zero-shot classification (no training data needed), compared against a trainable TF-IDF + Logistic Regression model.

In [ ]:
zero_shot = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')
categories = ['delivery', 'product quality', 'pricing', 'customer support', 'other']

out = zero_shot(texts[0], candidate_labels=categories)
print(out['labels'][0], out['scores'][0])

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

product quality 0.5000036358833313


## Summary

The trainable classifier (`TfidfCategoryClassifier`) was trained on a hand-labelled dataset of 125 examples and evaluated with a held-out test split. See `src/classification.py` and `src/train_classifier.py` for the final implementation, and `data/category_training_set.csv` for the training data.